# Supplementary Figure S1 — NbBench feature distributions by label

Generates **Supplementary Figure S1** for PEDS-26-0067: distributions of the ten
top-ranked univariate features from Figure 2, computed on the **NbBench PolyRx**
dataset (the same data as Figure 2), colored by polyreactivity label.

**How to run:** Runtime → Run all. CPU runtime is fine (no GPU). ~2 minutes.
Outputs `figureS1_feature_distributions.{tif,pdf,png}`; download the `.png` and send
it back for embedding, or upload the `.tif` (350 dpi) directly to ScholarOne.

In [ ]:
!pip install -q datasets

In [ ]:
# Cell 1 — load NbBench and show its actual columns (so we map fields correctly)
import pandas as pd, numpy as np
from datasets import load_dataset

ds = load_dataset("ZYMScott/polyreaction")
print("splits:", {k: ds[k].num_rows for k in ds})
val = ds["validation"].to_pandas()
print("columns:", list(val.columns))
print(val.head(2).to_string())

In [ ]:
# Cell 2 — feature code (inline; identical definitions to the main analysis)
KYTE_DOOLITTLE = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
CHARGE_AT_PH74 = {"D":-1,"E":-1,"K":1,"R":1,"H":0.1}
POSITIVE=set("KR"); NEGATIVE=set("DE")
PKA={"C_term":3.55,"D":4.05,"E":4.45,"H":5.98,"K":10.0,"R":12.0,"Y":10.0,"C":9.0,"N_term":8.0}

def net_charge(s):
    if not isinstance(s,str) or not s: return np.nan
    return sum(CHARGE_AT_PH74.get(a,0) for a in s.upper())
def frac(s,sub):
    if not isinstance(s,str) or not s: return np.nan
    s=s.upper(); return sum(1 for a in s if a in sub)/len(s)
def frac_res(s,r):
    if not isinstance(s,str) or not s: return np.nan
    return s.upper().count(r)/len(s)
def estimate_pI(s):
    if not isinstance(s,str) or not s: return np.nan
    s=s.upper()
    def c_at(ph):
        c=1/(1+10**(ph-PKA["N_term"]))-1/(1+10**(PKA["C_term"]-ph))
        for a in s:
            if a in ("K","R"): c+=1/(1+10**(ph-PKA[a]))
            elif a in ("D","E"): c-=1/(1+10**(PKA[a]-ph))
            elif a=="H": c+=1/(1+10**(ph-PKA["H"]))
            elif a=="Y": c-=1/(1+10**(PKA["Y"]-ph))
            elif a=="C": c-=1/(1+10**(PKA["C"]-ph))
        return c
    lo,hi=0.0,14.0
    for _ in range(50):
        m=(lo+hi)/2
        if c_at(m)>0: lo=m
        else: hi=m
    return (lo+hi)/2
print("feature code ready")

In [ ]:
# Cell 3 — map NbBench columns defensively, then compute the top-10 features
# The NbBench PolyRx frame provides CDR fields and a full-sequence field; column
# names can vary, so we resolve them here rather than assuming.
def pick(cols, *candidates):
    for c in candidates:
        if c in cols: return c
    raise KeyError(f"none of {candidates} found in columns {list(cols)}")

cols = set(val.columns)
COL_FULL  = pick(cols, "seq", "sequence", "full_seq", "VHH", "vhh_seq")
COL_H2    = pick(cols, "CDR2_nogaps", "cdr2", "CDRH2", "cdrh2")
COL_H3    = pick(cols, "CDR3_nogaps", "cdr3", "CDRH3", "cdrh3")
COL_LABEL = pick(cols, "label", "labels", "y", "polyreactive")
print("using:", dict(full=COL_FULL, H2=COL_H2, H3=COL_H3, label=COL_LABEL))

# keep rows with a non-empty CDR-H3 (matches the main analysis)
val = val[val[COL_H3].fillna("").astype(str).str.len() > 0].reset_index(drop=True)

full = val[COL_FULL].fillna("").astype(str)
h2   = val[COL_H2].fillna("").astype(str)
h3   = val[COL_H3].fillna("").astype(str)

X = pd.DataFrame(index=val.index)
X["full_pI"]         = full.apply(estimate_pI)
X["full_charge"]     = full.apply(net_charge)
X["full_abs_charge"] = X["full_charge"].abs()
X["full_R"]          = full.apply(lambda x: frac_res(x,"R"))
X["full_pos_frac"]   = full.apply(lambda x: frac(x, POSITIVE))
X["full_neg_frac"]   = full.apply(lambda x: frac(x, NEGATIVE))
X["H3_charge"]       = h3.apply(net_charge)
X["H3_pI"]           = h3.apply(estimate_pI)
X["H2_charge"]       = h2.apply(net_charge)
X["H3_neg_frac"]     = h3.apply(lambda x: frac(x, NEGATIVE))
X["label"]           = val[COL_LABEL].astype(int).values
N=len(X); n_pos=int((X.label==1).sum()); n_neg=int((X.label==0).sum())
print(f"features computed on n={N} ({n_pos} polyreactive, {n_neg} non-polyreactive)")

In [ ]:
# Cell 4 — plot the 10-panel distribution grid and save at 350 dpi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.titlesize":11,
                     "axes.spines.top":False,"axes.spines.right":False})

feats = [
    ("full_pI",         "Full-seq pI  (AUROC 0.779)"),
    ("full_charge",     "Full-seq net charge  (0.779)"),
    ("full_abs_charge", "Full-seq |charge|  (0.764)"),
    ("full_R",          "Full-seq Arg fraction  (0.737)"),
    ("full_pos_frac",   "Full-seq positive fraction  (0.734)"),
    ("full_neg_frac",   "Full-seq negative fraction  (0.732)"),
    ("H3_charge",       "CDR-H3 net charge  (0.726)"),
    ("H3_pI",           "CDR-H3 pI  (0.722)"),
    ("H2_charge",       "CDR-H2 net charge  (0.719)"),
    ("H3_neg_frac",     "CDR-H3 negative fraction  (0.695)"),
]
C_POS, C_NEG = "#d62728", "#4c72b0"
fig, axes = plt.subplots(2, 5, figsize=(16, 6.4))
for ax,(col,title) in zip(axes.ravel(), feats):
    pos = X[X.label==1][col].dropna().values
    neg = X[X.label==0][col].dropna().values
    allv = np.concatenate([pos,neg])
    lo,hi = np.percentile(allv,1), np.percentile(allv,99)
    bins = np.linspace(lo,hi,40)
    ax.hist(neg,bins=bins,color=C_NEG,alpha=0.55,density=True)
    ax.hist(pos,bins=bins,color=C_POS,alpha=0.55,density=True)
    ax.set_title(title,fontsize=9.5); ax.set_yticks([])
handles=[mpatches.Patch(color=C_NEG,alpha=0.55,label="Non-polyreactive"),
         mpatches.Patch(color=C_POS,alpha=0.55,label="Polyreactive")]
fig.legend(handles=handles,loc="upper center",bbox_to_anchor=(0.5,0.985),ncol=2,fontsize=10,frameon=True)
fig.suptitle(f"Distributions of the top-ranked univariate features (Figure 2) on the "
             f"NbBench PolyRx validation set\n(n = {N:,}: {n_pos:,} polyreactive, "
             f"{n_neg:,} non-polyreactive), colored by label", fontsize=12, y=1.06)
fig.tight_layout(rect=[0,0,1,0.93])
fig.savefig("figureS1_feature_distributions.pdf", bbox_inches="tight")
fig.savefig("figureS1_feature_distributions.tif", dpi=350, bbox_inches="tight", pil_kwargs={"compression":"tiff_lzw"})
fig.savefig("figureS1_feature_distributions.png", dpi=150, bbox_inches="tight")
print("saved figureS1_feature_distributions.{tif,pdf,png}")
plt.show()

Download `figureS1_feature_distributions.png` (folder icon on the left → download)
and send it back for embedding in the manuscript. The `.tif` is the 350 dpi
production file for ScholarOne.